# L19 Lab: Fox Hunt, notebook

This notebook is your fox-hunt script. You write four short functions, test them on a practice set in the Pre-Lab, and run the same notebook on your own readings in the lab to name the box.

**To run it:** save this notebook on your own laptop. It needs Python 3 with numpy and matplotlib, and Jupyter: `python -m pip install numpy matplotlib jupyterlab` installs all three. Start Jupyter with `jupyter lab`, open the notebook, and run the cells in order from the top.

**The frame.** $x$ runs along the tape and $y$ straight downrange, with the origin at the center of the table, on the middle station. Every station sits on the line ($y = 0$) with the kit facing downrange, so a station at $x_i$ that reads Angle $\theta_i$ puts the fox somewhere on the line $x = x_i + y\tan\theta_i$. The angles you type in already have your reference-shot sign $s$ applied, so a positive angle points toward the $+x$ end of the tape.

| When | What you do here |
| :-- | :-- |
| Pre-Lab (Part 1) | write the four functions, pass the tests, then run the practice set |
| In the lab (Part 2) | type in the posted boxes and your stations, and make the call |
| Writeup (Part 3) | solve again with only two stations |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Your four functions

Write these in the Pre-Lab. Each one is a few lines of numpy. A station is a tuple `(x_m, theta_deg, peak_gain_db)`; leave the gain as `None` if you did not record it. A box is `(name, x_m, y_m)`.

- **`solve`** finds the point closest to every bearing line at once, the least-squares fix of Lesson 19, "Many Bearings: Least Squares", and how far that point sits from each line (its residual).
- **`drop_worst`** finds the one station that disagrees with the rest and solves again without it.
- **`rank_boxes`** lists the boxes from nearest the fix to farthest.
- **`amplitude_check`** (optional) scores how well the Peak Gain readings fit a fox in a given box. You need it only when two boxes sit about equally close to the fix.

Until you write a function, running a cell that calls it stops with `NotImplementedError`. The plot below still draws the bearing lines without it.

In [ ]:
def solve(stations):
    """Fix the fox from every station's bearing.

    stations: list of (x_m, theta_deg, peak_gain_db or None)
    Returns (fix, residuals): fix is a length-2 array (x, y) in meters;
    residuals has one entry per station, in meters: the distance from the
    fix to that station's bearing line.
    """
    raise NotImplementedError("TODO (Pre-Lab)")

In [ ]:
def drop_worst(stations):
    """Find the one station whose bearing disagrees with the rest.

    Returns (index of that station, fix without it).
    """
    raise NotImplementedError("TODO (Pre-Lab)")

In [ ]:
def rank_boxes(fix, boxes):
    """Order the boxes from most to least likely to hold the fox.

    boxes: list of (name, x_m, y_m)
    Returns a list of (name, distance_m), nearest first.
    """
    raise NotImplementedError("TODO (Pre-Lab)")

In [ ]:
def amplitude_check(stations, box_xy):
    """Score how well the Peak Gain readings fit a fox at box_xy.

    Smaller is a better fit. Used only to choose between two boxes that
    sit about equally close to the fix.
    """
    raise NotImplementedError("TODO (Pre-Lab, optional)")

## Test your functions

Run this cell each time you change a function. It makes up a fox at a known spot, works out the exact bearing and level each station would read, and checks your functions against them, so every answer is exact. It prints PASS or FAIL for each check, with a hint when one fails. Pass them all before you move on to the practice set.

In [ ]:
def test_functions():
    """Check the four functions on a made-up hunt with a known answer."""
    true_xy = np.array([0.40, 1.50])
    xs = [-1.5, -0.75, 0.0, 0.75, 1.5]

    def reading(x, xy, bend=0.0):
        dx, dy = xy[0] - x, xy[1]
        theta = np.degrees(np.arctan2(dx, dy))
        level = -20 * np.log10(np.hypot(dx, dy)) + 12 * np.log10(np.cos(np.radians(theta))) - 5
        return (x, round(theta + bend, 3), round(level, 2))

    exact = [reading(x, true_xy) for x in xs]
    bad = 1                                  # the station at -0.75 m reads 8 degrees off
    bent = [reading(x, true_xy, 8.0 if i == bad else 0.0) for i, x in enumerate(xs)]

    def check(name, ok, hint):
        print(f"{'PASS' if ok else 'FAIL'}  {name}" + ("" if ok else f"\n      hint: {hint}"))

    def run(name, f):
        try:
            f()
        except NotImplementedError:
            print(f"----  {name}: not written yet")
        except Exception as e:
            print(f"FAIL  {name}: stopped with {type(e).__name__}: {e}")

    def t_solve():
        fix, res = solve(exact)
        fix = np.asarray(fix, dtype=float)
        if fix[1] < 0:
            hint = ("the fix is behind the firing line (y < 0): with a positive angle "
                    "toward +x, each bearing is the line x = x_i + y tan(theta_i); "
                    "check the sign of the tan term")
        else:
            hint = ("each bearing is the line x = x_i + y tan(theta_i); "
                    "solve all of them together by least squares")
        found = np.linalg.norm(fix - true_xy) < 0.01
        check(f"solve finds the fox: fix ({fix[0]:+.2f}, {fix[1]:+.2f}) m, true (+0.40, +1.50) m",
              found, hint)
        if not found:
            print("----  the residual checks wait until solve finds the fox")
            return
        check("solve returns one residual per station", len(res) == len(exact),
              f"got {len(res)} residuals for {len(exact)} stations")
        check("residuals are near zero for exact bearings",
              len(res) == len(exact) and np.max(np.abs(res)) < 0.01,
              "a residual is the distance from the fix to each station's line")
        fix_b, res_b = solve(bent)
        fix_b = np.asarray(fix_b, dtype=float)
        res_b = np.abs(np.asarray(res_b, dtype=float))
        # each residual should be the perpendicular distance from the fix to that line
        dist = np.array([abs((fix_b[0] - x) * np.cos(np.radians(th)) - fix_b[1] * np.sin(np.radians(th)))
                         for x, th, _ in bent])
        check("residuals are distances from the fix to each line, in meters",
              len(res_b) == len(bent) and np.allclose(res_b, dist, atol=0.001),
              "for one bad bearing, your residuals were "
              + ", ".join(f"{r:.3f}" for r in res_b) + " but the distances are "
              + ", ".join(f"{d:.3f}" for d in dist)
              + " m. Not angles, not squared: the perpendicular distance to each line")
        check("the bad bearing has the largest residual", int(np.argmax(res_b)) == bad,
              f"largest residual at station {xs[int(np.argmax(res_b))]:+.2f} m, expected -0.75 m")

    def t_drop():
        k, fix = drop_worst(bent)
        fix = np.asarray(fix, dtype=float)
        check("drop_worst drops the bad station (-0.75 m)", k == bad,
              f"dropped index {k}; it should drop the station with the largest residual")
        check("drop_worst's fix lands on the fox", np.linalg.norm(fix - true_xy) < 0.01,
              "solve again with every station except the one dropped "
              "(if solve fails above, fix it first)")

    boxes = [("A", -0.60, 1.20), ("B", 0.40, 1.50), ("C", 0.15, 1.25), ("D", 1.00, 2.00)]

    def t_rank():
        ranked = rank_boxes(true_xy, boxes)
        names = [r[0] for r in ranked]
        check("rank_boxes puts the nearest box first", names == ["B", "C", "D", "A"],
              f"got {names}; sort by distance from the fix, nearest first")
        dists = {r[0]: r[1] for r in ranked}
        check("rank_boxes gives distances in meters", abs(dists.get("C", 0) - 0.3536) < 0.001,
              f"box C is 0.354 m from the fix; got {dists.get('C')}")

    def t_amp():
        sb = amplitude_check(exact, (0.40, 1.50))
        sc = amplitude_check(exact, (0.15, 1.25))
        check(f"amplitude_check prefers the right box (B {sb:.2f}, C {sc:.2f})", sb < sc,
              "a smaller score should mean a better fit; compare the levels along "
              "the line with free-space spreading and the element roll-off")
        louder = [(x, th, g + 7) for x, th, g in exact]
        check("amplitude_check ignores the overall level",
              abs(amplitude_check(louder, (0.40, 1.50)) - sb) < 0.01,
              "raising every level by 7 dB changed the score; fit out the overall "
              "level (the fox's power and your Rx Gain are unknown), and compare only the shape")

    run("solve", t_solve)
    run("drop_worst", t_drop)
    run("rank_boxes", t_rank)
    run("amplitude_check (optional)", t_amp)

test_functions()

## Reading and drawing

Run this cell; you do not need to change it. `report` runs your functions and prints the results. `draw` plots the stations, their bearing lines, the boxes, and the fix: the lines should cross near one box, and a bad bearing is the line that misses the crossing.

In [ ]:
def report(stations, boxes):
    """Run the four functions and print the fix, the residuals, and the ranking."""
    fix, res = solve(stations)
    print(f"Fix from {len(stations)} bearings: x = {fix[0]:+.2f} m, y = {fix[1]:+.2f} m")
    for (x, th, _), r in zip(stations, res):
        print(f"  station x = {x:+.2f} m, angle {th:+6.1f} deg: residual {r:+.2f} m")
    if len(stations) >= 4:
        k, fix2 = drop_worst(stations)
        print(f"Without station x = {stations[k][0]:+.2f} m: "
              f"fix ({fix2[0]:+.2f}, {fix2[1]:+.2f}) m")
    print("Boxes, most likely first:")
    ranked = rank_boxes(fix, boxes)
    for name, dist in ranked:
        print(f"  box {name}: {dist:.2f} m from the fix")
    if len(ranked) >= 2 and ranked[1][1] - ranked[0][1] < 0.15:
        where = {b[0]: (b[1], b[2]) for b in boxes}
        print("The two nearest boxes are close; amplitude scores (smaller fits better):")
        for name, _ in ranked[:2]:
            try:
                print(f"  box {name}: {amplitude_check(stations, where[name]):.2f}")
            except NotImplementedError:
                print("  amplitude_check is not written yet")
                break
    return fix


def draw(stations, boxes, title=""):
    """Plot the range from above: stations, bearing lines, boxes, and the fix."""
    fig, ax = plt.subplots(figsize=(7, 4.5))
    y = np.array([0.0, 2.6])
    for x, th, _ in stations:
        ax.plot(x + y * np.tan(np.radians(th)), y, lw=1, color="C0", alpha=0.7)
        ax.plot(x, 0, "^", color="C0", ms=9)
        ax.annotate(f"{th:+.1f}°", (x, 0), textcoords="offset points",
                    xytext=(0, -16), ha="center", fontsize=9)
    for name, bx, by in boxes:
        ax.add_patch(plt.Rectangle((bx - 0.1, by - 0.1), 0.2, 0.2,
                                   fill=False, color="0.4"))
        ax.annotate(str(name), (bx, by), ha="center", va="center", fontsize=10)
    try:
        fix, _ = solve(stations)
        ax.plot(*fix, "x", color="C3", ms=12, mew=2, label="fix")
        ax.legend(loc="upper right")
    except NotImplementedError:
        pass
    ax.set_xlim(-2.0, 2.0); ax.set_ylim(-0.3, 2.6)
    ax.set_aspect("equal")
    ax.set_xlabel("x along the tape (m)"); ax.set_ylabel("y downrange (m)")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    plt.show()

## Pre-Lab: the practice set

A practice set from an earlier class, recorded at one Rx Gain, with the boxes posted that day. This is the set in Question 3 of the packet.

1. Run the cell with every station in, and look at the plot. Decide which stations to use, and comment out any you leave out (put `#` at the start of its line). Say why in the packet.
2. Run it again on the stations you kept, and record the fix, the residuals, the station `drop_worst` drops, and the ranking in Question 3.

**Check:** with the stations you should keep, the fix lands near box 3 and the largest residual is about $0.15\ \text{m}$; with the outlier dropped, every residual left is a few centimeters.

In [ ]:
PRACTICE_STATIONS = [
    # x_m,   angle_deg, peak_gain_db
    (-1.5,   +52.6,     -19.6),
    (-0.75,  +38.2,     -15.0),
    ( 0.0,   +27.2,     -10.7),
    (+0.75,   -5.8,     -10.0),
    (+1.5,   -31.4,     -13.5),
]
PRACTICE_BOXES = [
    # name, x_m,  y_m
    ("1", -0.75, 1.20),
    ("2",  0.00, 1.00),
    ("3",  0.55, 1.60),
    ("4", -0.40, 1.90),
    ("5",  1.00, 0.90),
]

draw(PRACTICE_STATIONS, PRACTICE_BOXES, "practice set")
report(PRACTICE_STATIONS, PRACTICE_BOXES);

## Two stations at a time

The box in the practice set was box 3. Solve with only two stations to see what the baseline does to the fix: the two ends you kept, then two neighbors. Record each fix and its distance from box 3 in the last part of Question 3. On your own data, do the same for Part 3 of the packet, using your own stations and the live box.

**Check:** the wide pair lands within a few centimeters of the box; the neighbors miss by several times more, almost all of it in depth.

In [ ]:
def pick(stations, *xs):
    """The stations at the given positions along the tape."""
    return [s for s in stations if any(abs(s[0] - x) < 0.01 for x in xs)]

LIVE = np.array([0.55, 1.60])     # box 3; on your own data, the live box
PAIRS = {"wide": (-0.75, +1.5), "narrow": (+0.75, +1.5)}

for name, xs in PAIRS.items():
    fix, _ = solve(pick(PRACTICE_STATIONS, *xs))
    print(f"{name}: fix ({fix[0]:+.2f}, {fix[1]:+.2f}) m, "
          f"{np.linalg.norm(fix - LIVE):.2f} m from the box")

## In the lab: your hunt

1. When the instructor posts the boxes, type them into `BOXES`. Add or remove rows to match: there may be more or fewer boxes than in the practice set.
2. At each station, type in $s$ times the mean of your ten Angle readings, and the Peak Gain. For a station with no clear tone, comment its row out.
3. Run the cell, look at the plot, and make the call before the reveal. Record the fix, the largest residual, any station dropped, and the box you name in the packet.

In [ ]:
BOXES = [
    # name, x_m, y_m   (posted by the instructor), for example:
    # ("1", -0.75, 1.20),
]
STATIONS = [
    # x_m, s * mean angle_deg, peak_gain_db, for example:
    # (-1.5, +12.3, -14.0),
]

if len(STATIONS) >= 2 and BOXES:
    draw(STATIONS, BOXES, "our hunt")
    report(STATIONS, BOXES);
else:
    print("Type in the boxes and at least two stations, then run this cell again.")

## Before you close

Save the notebook with your readings and plots, and attach it to the packet; it is the code the packet asks for. Keep it for the writeup: Part 3 re-solves your own stations two at a time with the cell above.